# Kinship Library
## Notebook 01: Collection, Data Quality and Exploratory Analysis

### Goal

Build a trustworthy thematic book catalogue before modeling.

This notebook covers the useful work from the original Day 1 and Day 2.

### Main project question

Can book descriptions and subject metadata help readers discover meaningful books about living in connection with self, community, nature, and ancestral knowledge?

### Data principles

We keep source metadata separate from our later reader paths.

We do not assume that Yoga, Ayurveda, Indian spirituality, Kogi knowledge, Muisca knowledge, and ecological philosophy are interchangeable.

We keep the source and search query for every record so that the collection process remains traceable.

### Outputs

`data/raw/open_library_books.csv`

`data/raw/books_to_scrape.csv`

`data/processed/books_clean.csv`

## 1. Setup

### Purpose

Import simple libraries and define repository relative paths.

### Expected result

The folders are created automatically. The same notebook can run in Colab or from a cloned repository.

In [ ]:
from pathlib import Path
import time
import re
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from bs4 import BeautifulSoup
from scipy import stats

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DATA = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA = PROJECT_ROOT / "data" / "processed"

RAW_DATA.mkdir(parents=True, exist_ok=True)
PROCESSED_DATA.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

## 2. Business understanding

### Reader need

Kinship Library is designed for readers who want to explore questions of ecological connection, meaning, embodied practice, Yoga philosophy, Ayurveda, Indian spirituality, and selected Indigenous knowledge traditions from South America.

Later, a reader may describe an experience such as fear of the future, ecological concern, disconnection from nature, or a search for meaning.

The recommender will not diagnose that person. It will retrieve books whose themes are semantically related to the reader's self described interest.

### Questions for this phase

1. Can we collect enough relevant books with meaningful descriptions and subjects?
2. Which sources provide the most complete metadata?
3. Which themes are overrepresented or underrepresented?
4. How much missing information do we have?
5. Are there extreme values or suspicious records that need inspection?

## 3. Search vocabulary

These queries are collection tools, not labels about what a book can psychologically help with.

In [ ]:
THEME_QUERIES = {
    "humans_and_nature": [
        "ecology human nature",
        "environmental humanities",
        "more than human",
        "ecofeminism",
        "deep ecology",
        "human nature connection"
    ],
    "yoga_philosophy": [
        "Yoga Sutras Patanjali",
        "Vedas",
        "Upanishads",
        "Bhagavad Gita",
        "yoga philosophy"
    ],
    "yoga_way_of_life": [
        "eight limbs yoga",
        "pranayama yoga",
        "meditation yoga",
        "yoga way of life"
    ],
    "ayurveda_indian_spirituality": [
        "Ayurveda",
        "Vedanta",
        "Indian spirituality",
        "Autobiography of a Yogi",
        "Indian Tantra"
    ],
    "south_american_indigenous_knowledge": [
        "Kogi Colombia",
        "Muisca Colombia",
        "Andean indigenous knowledge",
        "Colombia indigenous ecology",
        "South America indigenous cosmology"
    ]
}

SEED_TITLES = [
    "Staying with the Trouble",
    "Ancestral Future",
    "The Yoga Sutras of Patanjali",
    "Autobiography of a Yogi",
    "Bhagavad Gita"
]

## 4. Open Library collection

### Purpose

Use the Open Library Search API to collect multiple books per request.

### Important

Replace the contact email below with your own contact email before making repeated requests.

The code caches the result as CSV so we do not call the API again every time we rerun the notebook.

In [ ]:
OPEN_LIBRARY_FILE = RAW_DATA / "open_library_books.csv"

if OPEN_LIBRARY_FILE.exists():
    open_library_books = pd.read_csv(OPEN_LIBRARY_FILE)
    print("Loaded cached Open Library data:", len(open_library_books))

else:
    rows = []

    headers = {
        "User-Agent": "KinshipLibraryStudentProject your_email@example.com"
    }

    for collection_theme, queries in THEME_QUERIES.items():
        for query in queries:

            params = {
                "q": query,
                "limit": 30,
                "fields": (
                    "key,title,author_name,first_publish_year,"
                    "subject,isbn,language,ratings_average,ratings_count"
                )
            }

            response = requests.get(
                "https://openlibrary.org/search.json",
                params=params,
                headers=headers,
                timeout=30
            )

            if response.status_code != 200:
                print("Skipped:", query, "Status:", response.status_code)
                continue

            data = response.json()

            for book in data.get("docs", []):
                rows.append({
                    "source_id": book.get("key"),
                    "title": book.get("title"),
                    "author": ", ".join(book.get("author_name", [])),
                    "description": np.nan,
                    "subjects": ", ".join(book.get("subject", [])[:15]),
                    "published_year": book.get("first_publish_year"),
                    "average_rating": book.get("ratings_average"),
                    "ratings_count": book.get("ratings_count"),
                    "isbn": book.get("isbn", [np.nan])[0] if book.get("isbn") else np.nan,
                    "language": book.get("language", [np.nan])[0] if book.get("language") else np.nan,
                    "source": "open_library",
                    "collection_theme": collection_theme,
                    "search_query": query
                })

            time.sleep(1)

    open_library_books = pd.DataFrame(rows)
    open_library_books.to_csv(OPEN_LIBRARY_FILE, index=False)

    print("Saved Open Library data:", len(open_library_books))

open_library_books.head()

## 5. Books to Scrape sample

### Purpose

Demonstrate web scraping with a site built for scraping practice.

This sample is kept separate from the thematic model because it does not provide the rich descriptions we need for semantic recommendation.

In [ ]:
SCRAPED_FILE = RAW_DATA / "books_to_scrape.csv"

if SCRAPED_FILE.exists():
    scraped_books = pd.read_csv(SCRAPED_FILE)
    print("Loaded cached scraped data:", len(scraped_books))

else:
    rating_map = {
        "One": 1,
        "Two": 2,
        "Three": 3,
        "Four": 4,
        "Five": 5
    }

    rows = []

    for page in range(1, 6):
        url = f"https://books.toscrape.com/catalogue/page-{page}.html"
        response = requests.get(url, timeout=30)

        if response.status_code != 200:
            print("Skipped page:", page)
            continue

        soup = BeautifulSoup(response.text, "html.parser")

        for book in soup.select("article.product_pod"):
            rating_word = book.p.get("class", ["", ""])[1]

            rows.append({
                "title": book.h3.a["title"],
                "price": book.select_one(".price_color").get_text(strip=True),
                "rating": rating_map.get(rating_word),
                "availability": book.select_one(".availability").get_text(strip=True),
                "source": "books_to_scrape"
            })

        time.sleep(1)

    scraped_books = pd.DataFrame(rows)
    scraped_books.to_csv(SCRAPED_FILE, index=False)

scraped_books.head()

## 6. Google Books status

Our first Colab test returned HTTP 429 even for one request.

For reproducibility, this notebook does not depend on Google Books collection. We will solve authentication or quota separately and use it later only as optional enrichment.

This prevents one external service from blocking the entire project.

## 7. Clean the thematic catalogue

### Purpose

Standardize types, remove obvious duplicates, inspect missing values, and create simple quality features.

In [ ]:
books = open_library_books.copy()

books["title"] = books["title"].astype("string").str.strip()
books["author"] = books["author"].astype("string").str.strip()
books["subjects"] = books["subjects"].astype("string").str.strip()

books["published_year"] = pd.to_numeric(
    books["published_year"],
    errors="coerce"
)

books["average_rating"] = pd.to_numeric(
    books["average_rating"],
    errors="coerce"
)

books["ratings_count"] = pd.to_numeric(
    books["ratings_count"],
    errors="coerce"
)

books = books.dropna(subset=["title"])

books = books.drop_duplicates(
    subset=["title", "author"],
    keep="first"
)

books = books.reset_index(drop=True)

books["subject_count"] = (
    books["subjects"]
    .fillna("")
    .apply(lambda text: 0 if text == "" else len(text.split(",")))
)

books.info()

## 8. Missing values

In [ ]:
missing_values = pd.DataFrame({
    "missing_count": books.isnull().sum(),
    "missing_percent": (books.isnull().mean() * 100).round(2)
}).sort_values("missing_percent", ascending=False)

missing_values

## 9. Univariate exploration

In [ ]:
plt.figure(figsize=(8, 5))
books["average_rating"].dropna().hist(bins=15)
plt.title("Distribution of average ratings")
plt.xlabel("Average rating")
plt.ylabel("Number of books")
plt.show()

plt.figure(figsize=(8, 5))
books["subject_count"].hist(bins=20)
plt.title("Distribution of subject count")
plt.xlabel("Number of subjects")
plt.ylabel("Number of books")
plt.show()

## 10. Outlier check

We use the IQR rule to flag unusual values. Flagging an outlier does not mean deleting it.

In [ ]:
q1 = books["subject_count"].quantile(0.25)
q3 = books["subject_count"].quantile(0.75)
iqr = q3 - q1

lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

subject_outliers = books[
    (books["subject_count"] < lower_bound)
    | (books["subject_count"] > upper_bound)
]

print("LCV for IQR inspection:", lower_bound)
print("UCV for IQR inspection:", upper_bound)
print("Outliers:", len(subject_outliers))

plt.figure(figsize=(8, 4))
plt.boxplot(books["subject_count"].dropna(), vert=False)
plt.title("Subject count outlier check")
plt.xlabel("Number of subjects")
plt.show()

## 11. Probability

Estimate the probability that a randomly selected record in our collected sample has at least five subjects.

This probability describes our sample, not the entire population of books.

In [ ]:
has_rich_subject_metadata = books["subject_count"] >= 5
probability_rich_metadata = has_rich_subject_metadata.mean()

print(
    "Estimated probability:",
    round(probability_rich_metadata, 3)
)

## 12. Bivariate exploration and correlation

In [ ]:
correlation_data = books[
    ["average_rating", "ratings_count", "published_year", "subject_count"]
].copy()

correlation_matrix = correlation_data.corr(numeric_only=True)
correlation_matrix

## 13. Normal distribution check for ratings

Null hypothesis: average ratings are compatible with a normal distribution.

Alternative hypothesis: average ratings are not compatible with a normal distribution.

Alpha level: 0.05.

In [ ]:
rating_values = books["average_rating"].dropna()

if len(rating_values) >= 3:
    sample = rating_values.sample(
        min(len(rating_values), 500),
        random_state=RANDOM_STATE
    )

    statistic, p_value = stats.shapiro(sample)

    print("Test statistic:", round(statistic, 4))
    print("P value:", round(p_value, 4))
    print("Alpha:", 0.05)

    if p_value < 0.05:
        print("Decision: reject the null hypothesis")
    else:
        print("Decision: fail to reject the null hypothesis")
else:
    print("Not enough rating observations")

## 14. Bivariate hypothesis test with rejection regions

Question: Is mean subject count different between the humans and nature collection and Yoga philosophy collection?

Null hypothesis: the two population means are equal.

Alternative hypothesis: the two population means are different.

Test: two sided Welch t test.

Alpha: 0.05.

In [ ]:
group_a = books.loc[
    books["collection_theme"] == "humans_and_nature",
    "subject_count"
].dropna()

group_b = books.loc[
    books["collection_theme"] == "yoga_philosophy",
    "subject_count"
].dropna()

alpha = 0.05

if len(group_a) >= 2 and len(group_b) >= 2:

    test_statistic, p_value = stats.ttest_ind(
        group_a,
        group_b,
        equal_var=False
    )

    var_a = group_a.var(ddof=1)
    var_b = group_b.var(ddof=1)
    n_a = len(group_a)
    n_b = len(group_b)

    df_num = (var_a / n_a + var_b / n_b) ** 2
    df_den = (
        ((var_a / n_a) ** 2) / (n_a - 1)
        + ((var_b / n_b) ** 2) / (n_b - 1)
    )
    degrees_freedom = df_num / df_den

    lcv = stats.t.ppf(alpha / 2, degrees_freedom)
    ucv = stats.t.ppf(1 - alpha / 2, degrees_freedom)

    print("Test statistic:", round(test_statistic, 4))
    print("P value:", round(p_value, 4))
    print("Alpha:", alpha)
    print("LCV:", round(lcv, 4))
    print("UCV:", round(ucv, 4))

    if p_value < alpha:
        print("Decision: reject the null hypothesis")
    else:
        print("Decision: fail to reject the null hypothesis")

    x = np.linspace(
        stats.t.ppf(0.001, degrees_freedom),
        stats.t.ppf(0.999, degrees_freedom),
        500
    )
    y = stats.t.pdf(x, degrees_freedom)

    plt.figure(figsize=(9, 5))
    plt.plot(x, y)

    plt.fill_between(
        x[x <= lcv],
        stats.t.pdf(x[x <= lcv], degrees_freedom),
        alpha=0.3
    )

    plt.fill_between(
        x[x >= ucv],
        stats.t.pdf(x[x >= ucv], degrees_freedom),
        alpha=0.3
    )

    plt.axvline(lcv, linestyle="--", label="LCV")
    plt.axvline(ucv, linestyle="--", label="UCV")
    plt.axvline(test_statistic, label="Test statistic")

    plt.title("Welch t test with rejection regions")
    plt.xlabel("t value")
    plt.ylabel("Probability density")
    plt.legend()
    plt.show()

else:
    print("Not enough observations for the test")

## 15. Save the clean catalogue

The next notebook will use this file. This separation means we do not need to call the APIs every time we experiment with NLP.

In [ ]:
CLEAN_FILE = PROCESSED_DATA / "books_clean.csv"

books.to_csv(CLEAN_FILE, index=False)

print("Saved:", CLEAN_FILE)
print("Rows:", len(books))